# Getting started with `grangercm`

This vignette walks through the full workflow: preparing series for
stationarity, building a **Granger Causality Matrix**, correcting for
multiple testing, visualising the result as a heatmap, and running the
nonlinear backend.

We use a synthetic 4-variable system with a *known* causal structure so
you can check the answers:

```
x : exogenous noise
y : driven by x at lag 1 (linear)
w : driven by x at lag 2 through a nonlinear (tanh) channel
u : independent noise
```


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
n = 600
x = rng.normal(size=n)
y = np.zeros(n)
w = np.zeros(n)
for t in range(2, n):
    y[t] = 0.6 * x[t - 1] + 0.2 * y[t - 1] + rng.normal(scale=0.3)
    w[t] = 0.9 * np.tanh(2 * x[t - 2]) + 0.1 * w[t - 1] + rng.normal(scale=0.3)
u = rng.normal(size=n)

# A random walk, to show the stationarity step doing real work.
level = np.cumsum(rng.normal(size=n))
df = pd.DataFrame({'x': x, 'y': y, 'w': w, 'u': u, 'level': level})
df.head()

,x,y,w,u,level
0,0.125730,0.000000,0.000000,1.429691,0.343305
1,-0.132105,0.000000,0.000000,1.063556,1.420987
2,0.640423,-0.426312,-0.187802,1.245170,2.074768
3,0.104900,0.229623,0.432064,-0.512890,1.460773
4,-0.535669,0.192127,1.043121,0.244001,2.984889


## 1. Check and enforce stationarity

Granger tests assume weakly stationary inputs. `adf_report` runs the
Augmented Dickey-Fuller test on every column; `make_stationary`
differences until each series passes.

In [2]:
from grangercm import adf_report, make_stationary

print('Before:')
print(adf_report(df))

stat_df, orders = make_stationary(df, method='diff')
print('\nDifferencing orders applied:', orders)
print('\nAfter:')
print(adf_report(stat_df))

Before:


           adf_stat       p_value  stationary
variable                                     
x        -24.957596  0.000000e+00        True
y        -20.362455  0.000000e+00        True
w        -22.252785  0.000000e+00        True
u         -6.398617  2.021473e-08        True
level     -1.367235  5.978795e-01       False

Differencing orders applied: {'x': 0, 'y': 0, 'w': 0, 'u': 0, 'level': 1}

After:
           adf_stat       p_value  stationary
variable                                     
x        -24.935811  0.000000e+00        True
y        -20.345401  0.000000e+00        True
w        -22.234245  0.000000e+00        True
u         -6.413436  1.866310e-08        True
level    -24.293848  0.000000e+00        True


Only `level` needed differencing; the others were already stationary.

## 2. Build the linear Granger Causality Matrix

Entry `[effect, cause]` is the p-value for *cause Granger-causes effect*.
We test lags 1-4, aggregate per pair by the minimum p-value, and apply a
Benjamini-Hochberg correction across all off-diagonal cells.

In [3]:
from grangercm import granger_matrix

result = granger_matrix(
    stat_df,
    variables=['x', 'y', 'w', 'u'],
    maxlag=4,
    test='ssr_ftest',
    aggregate='min',
    correction='fdr_bh',
)
result.pvalues.round(4)

,x,y,w,u
x,NaN,0.6434,0.4003,0.4650
y,0.0000,NaN,0.5663,0.4156
w,0.0000,0.0000,NaN,0.2736
u,0.4939,0.4003,0.6434,NaN


The `x` column shows small p-values in the `y` and `w` rows -- exactly
the planted links. Note the linear test detects `x -> w` only weakly,
because that channel is nonlinear; we revisit it in section 5.

## 3. Significant edges

In [4]:
result.edges(alpha=0.05)

,cause,effect,p_value,statistic,lag
0,x,y,1.928399e-210,2412.704622,1.0
1,x,w,7.313182e-168,796.870969,2.0
2,y,w,3.525329e-113,417.236905,2.0


## 4. Visualise as a heatmap

The heatmap reads like a correlation table. Green = low p-value
(significant causality), centered on `alpha`.

In [5]:
ax = result.plot(alpha=0.05)
plt.savefig('docs/heatmap.png', dpi=110, bbox_inches='tight')
plt.show()
print('saved docs/heatmap.png')

saved docs/heatmap.png


## 5. Nonlinear backend

`x -> w` is a `tanh` channel that the linear test underrates. The
nonlinear matrix swaps the autoregression for an arbitrary estimator and
assesses significance with a block-permutation test. Here we use a small
random forest and a reduced permutation count for speed.

In [6]:
from grangercm import nonlinear_granger_matrix
from sklearn.ensemble import RandomForestRegressor

nl = nonlinear_granger_matrix(
    stat_df,
    variables=['x', 'w', 'u'],
    maxlag=[2],
    estimator=RandomForestRegressor(n_estimators=60, random_state=0),
    n_permutations=99,
    n_folds=4,
)
nl.pvalues.round(4)

,x,w,u
x,NaN,1.00,0.12
w,0.01,NaN,0.33
u,0.46,0.68,NaN


The nonlinear test confirms `x -> w` at lag 2 and leaves the noise
variable `u` unflagged in both directions.

## Caveats

- Granger causality is *predictive*, not structural, causality.
- `aggregate='min'` is conventional but anti-conservative; always pair it
  with a `correction`.
- The nonlinear backend is much more expensive; restrict `variables`,
  `maxlag`, or `n_permutations` for wide panels.